In [1]:
import sys
import os
sys.path.append("/mnt/projects/sne/Gohar/12.PROJECT_Surrogate_Dynamics/surrogate-dynamics/")

from tqdm import tqdm
import torch
from embeddings.resolver import get_encdec

In [2]:
%load_ext autoreload
%autoreload 2
from dataloaders.sequence import SequenceDataModule

dm_config = {
    "input_chunk_length": 1,
    "output_chunk_length": 0,
    "datareader_type": "XTC_CHUNKS_CG",
    "datareader_args": {
        "tprfile": "../data_generation/ala2/solvated/300K/initialization/tpr_initial.tpr",
        "xtcfile": "../data_generation/ala2/solvated/ensemble/run_1/mdmod.xtc",
        "selection": "(resname ALA or resname ACE or resname NME) and not element H",
        "cg_window": 1,
    },
    # "dataset_type": "GRAPH",
    "dataset_type": "GRAPH_LATENT",
    "dataset_args": {
        "encoder_name": "BGE",
        "encoder_ckpt": "../train_runs/encoder/bge/run_1/checkpoints/best.ckpt",
        "precompute_graphs": False,
    },
    "train_size": 1,
    "batch_size": 1,
    "validation_size": 0,
    "val_batch_size": 0,
    "num_workers": 1,
    "n_seq_per_sample": 1,
    "norm_type": "minmax",
    "verbose": False,
}



/mnt/projects/sne/Gohar/12.PROJECT_Surrogate_Dynamics/surrogate-dynamics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def get_ensemble_var(emsemble_path: str):
    data_points = []
    run_list = [a for a in os.listdir(emsemble_path) if a.startswith("run_") and os.path.isdir(os.path.join(emsemble_path, a))]
    for run_name in tqdm(run_list, leave=True):
        dm_config["datareader_args"]["xtcfile"] = os.path.join(emsemble_path, run_name, "mdmod.xtc")
        original_stdout = sys.stdout
        original_stderr = sys.stderr
        sys.stdout = open(os.devnull, 'w')
        sys.stderr = open(os.devnull, 'w')
        dm = SequenceDataModule(**dm_config)
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        data_point = dm.get_train_dataset()[:][0]
        data_points.append(data_point)
    result = torch.stack([torch.stack(sublist) for sublist in data_points])
    
    return result.var(dim=0).mean().item()

In [4]:
results = {}
for i in range(1, 31):
    ensemble = f"ensemble_{i}"
    # suppress output of get_ensemble_var
    var = get_ensemble_var(f"../data_generation/ala2/solvated/multiensemble/{ensemble}")
    results[ensemble] = var

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 500/500 [01:26<00:00,  5.77it/s]


In [5]:
for ensemble, var in results.items():
    print(f"{ensemble}: {var}")

ensemble_1: 0.251319020986557
ensemble_2: 0.45174747705459595
ensemble_3: 0.26521408557891846
ensemble_4: 0.3648746907711029
ensemble_5: 0.263079971075058
ensemble_6: 0.3380843997001648
ensemble_7: 0.26195305585861206
ensemble_8: 0.24637186527252197
ensemble_9: 0.23704317212104797
ensemble_10: 0.8197678923606873
ensemble_11: 0.23322762548923492
ensemble_12: 0.3268861174583435
ensemble_13: 0.3523988723754883
ensemble_14: 0.3619059920310974
ensemble_15: 0.27718937397003174
ensemble_16: 0.32734882831573486
ensemble_17: 0.2778366804122925
ensemble_18: 0.316574364900589
ensemble_19: 0.2178546041250229
ensemble_20: 0.35993385314941406
ensemble_21: 0.2613861560821533
ensemble_22: 0.54001384973526
ensemble_23: 0.24021616578102112
ensemble_24: 0.3741133511066437
ensemble_25: 0.3019919991493225
ensemble_26: 0.45104891061782837
ensemble_27: 0.26180171966552734
ensemble_28: 0.2229040414094925
ensemble_29: 0.24330133199691772
ensemble_30: 0.22611907124519348
